# Stress Prediction v9 — Blood Pressure Feature Refinement

이번 버전은 기존 최고 구조를 유지하면서 **혈압 파생변수만 정밀 비교**합니다.

비교 대상:

1. Baseline
2. Baseline + 혈압비
3. Baseline + 맥압
4. Baseline + 혈압비 + 맥압
5. Baseline + 평균동맥압
6. 수축기/이완기 혈압 제거 + 혈압비 + 맥압

이후 가장 좋은 혈압 피처 구성을 기준으로:

- Tree quantile
- Pair quantile
- Pair weight
- 반올림
- 최근접 이웃 override

를 다시 탐색하고 제출 후보를 생성합니다.


In [1]:
from __future__ import annotations

import os
import warnings
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "stress_score"
ID_COLUMN = "ID"

RANDOM_SEED = 42
N_SPLITS = 5

SCREEN_N_ESTIMATORS = 700
FINAL_N_ESTIMATORS = 1200

BASE_TREE_QUANTILE = 0.52

TREE_QUANTILES = [0.50, 0.52, 0.54]
PAIR_QUANTILES = [0.46, 0.48, 0.50]
PAIR_WEIGHTS = [0.15, 0.20, 0.25]
OVERRIDE_THRESHOLDS = [0.15, 0.18, 0.20, None]
ROUND_OPTIONS = [2, None]

WINSORIZE_COLUMNS = [
    "mean_working",
    "diastolic_blood_pressure",
    "cholesterol",
    "glucose",
]

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FALLBACK = Path("/mnt/data/train(2).csv")
TEST_FALLBACK = Path("/mnt/data/test(2).csv")

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


DATA_DIR: /Users/bitsaem/Desktop/stress_project/open
OUTPUT_DIR: /Users/bitsaem/Desktop/stress_project/8_6


## 1. 데이터 로드


In [2]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

if not train_path.exists() and TRAIN_FALLBACK.exists():
    train_path = TRAIN_FALLBACK

if not test_path.exists() and TEST_FALLBACK.exists():
    test_path = TEST_FALLBACK

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

y = train_raw[TARGET].reset_index(drop=True)

X_raw = (
    train_raw
    .drop(columns=[TARGET, ID_COLUMN])
    .reset_index(drop=True)
)

X_test_raw = (
    test_raw
    .drop(columns=[ID_COLUMN])
    .reset_index(drop=True)
)

numeric_raw_columns = X_raw.select_dtypes(
    include=["number"]
).columns.tolist()

print("Train shape:", train_raw.shape)
print("Test shape :", test_raw.shape)
display(train_raw.head())


Train shape: (3000, 18)
Test shape : (3000, 17)


,ID,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,smoke_status,medical_history,family_medical_history,sleep_pattern,edu_level,mean_working,stress_score
0,TRAIN_0000,F,72,161.49,58.47,279.84,165,100,143.35,0.87,moderate,ex-smoker,high blood pressure,diabetes,sleep difficulty,bachelors degree,NaN,0.63
1,TRAIN_0001,M,88,179.87,77.60,257.37,178,111,146.94,0.07,moderate,ex-smoker,NaN,diabetes,normal,graduate degree,NaN,0.83
2,TRAIN_0002,M,47,182.47,89.93,226.66,134,95,142.61,1.18,light,ex-smoker,NaN,NaN,normal,high school diploma,9.0,0.70
3,TRAIN_0003,M,69,185.78,68.63,206.74,158,92,137.26,0.48,intense,ex-smoker,high blood pressure,NaN,oversleeping,graduate degree,NaN,0.17
4,TRAIN_0004,F,81,164.63,71.53,255.92,171,116,129.37,0.34,moderate,ex-smoker,diabetes,diabetes,sleep difficulty,bachelors degree,NaN,0.36


## 2. Baseline 및 혈압 피처 구성


In [3]:
def safe_divide(
    numerator: pd.Series,
    denominator: pd.Series,
) -> pd.Series:
    return numerator / (denominator.abs() + 1e-6)


def add_baseline_features(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    height_m = result["height"] / 100.0

    result["bmi"] = safe_divide(
        result["weight"],
        height_m.pow(2),
    )

    result["cholesterol_glucose_ratio"] = safe_divide(
        result["cholesterol"],
        result["glucose"],
    )

    return result


def build_bp_feature_set(
    frame: pd.DataFrame,
    config_name: str,
) -> pd.DataFrame:
    result = add_baseline_features(frame)

    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    if config_name == "baseline":
        return result

    if config_name == "bp_ratio":
        result["blood_pressure_ratio"] = safe_divide(
            sbp,
            dbp,
        )

    elif config_name == "pulse_pressure":
        result["pulse_pressure"] = sbp - dbp

    elif config_name == "ratio_plus_pulse":
        result["blood_pressure_ratio"] = safe_divide(
            sbp,
            dbp,
        )
        result["pulse_pressure"] = sbp - dbp

    elif config_name == "mean_arterial_pressure":
        result["mean_arterial_pressure"] = (
            sbp + 2.0 * dbp
        ) / 3.0

    elif config_name == "replace_bp_with_ratio_pulse":
        result["blood_pressure_ratio"] = safe_divide(
            sbp,
            dbp,
        )
        result["pulse_pressure"] = sbp - dbp

        result = result.drop(
            columns=[
                "systolic_blood_pressure",
                "diastolic_blood_pressure",
            ],
            errors="ignore",
        )

    else:
        raise ValueError(
            f"알 수 없는 config_name: {config_name}"
        )

    return result


BP_CONFIGS = [
    "baseline",
    "bp_ratio",
    "pulse_pressure",
    "ratio_plus_pulse",
    "mean_arterial_pressure",
    "replace_bp_with_ratio_pulse",
]

for name in BP_CONFIGS:
    sample = build_bp_feature_set(
        X_raw.head(5),
        name,
    )
    print(name, sample.shape[1], "features")


baseline 18 features
bp_ratio 19 features
pulse_pressure 19 features
ratio_plus_pulse 20 features
mean_arterial_pressure 19 features
replace_bp_with_ratio_pulse 18 features


## 3. 공통 모델 함수


In [4]:
def compute_iqr_bounds(
    train_frame: pd.DataFrame,
    columns: list[str],
):
    bounds = {}

    for col in columns:
        q1, q3 = train_frame[col].quantile(
            [0.25, 0.75]
        )
        iqr = q3 - q1

        bounds[col] = (
            q1 - 1.5 * iqr,
            q3 + 1.5 * iqr,
        )

    return bounds


def apply_winsorize(
    frame: pd.DataFrame,
    bounds,
):
    result = frame.copy()

    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(
            lower,
            upper,
        )

    return result


def add_feature_copies(
    frame: pd.DataFrame,
):
    result = frame.copy()

    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        if feature not in result.columns:
            continue

        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]

    return result


def make_preprocessor(
    frame: pd.DataFrame,
):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    numerical = frame.select_dtypes(
        include=["number"]
    ).columns.tolist()

    return ColumnTransformer([
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="missing",
                    ),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical,
        ),
        (
            "numerical",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
            numerical,
        ),
    ])


def tree_prediction_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
    n_estimators: int,
    return_importance: bool = False,
):
    # 기존 최고 구조 유지
    train_input = train_features.drop(
        columns=["gender"],
        errors="ignore",
    )
    val_input = val_features.drop(
        columns=["gender"],
        errors="ignore",
    )

    train_input = add_feature_copies(train_input)
    val_input = add_feature_copies(val_input)

    preprocessor = make_preprocessor(train_input)

    train_matrix = preprocessor.fit_transform(
        train_input
    )
    val_matrix = preprocessor.transform(
        val_input
    )

    model = ExtraTreesRegressor(
        n_estimators=n_estimators,
        min_samples_leaf=1,
        max_features=1,
        criterion="squared_error",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )

    model.fit(
        train_matrix,
        target,
    )

    prediction_matrix = np.column_stack([
        tree.predict(val_matrix)
        for tree in model.estimators_
    ])

    if not return_importance:
        return prediction_matrix

    try:
        feature_names = preprocessor.get_feature_names_out()

        importance_df = pd.DataFrame({
            "feature": feature_names,
            "importance": model.feature_importances_,
        }).sort_values(
            "importance",
            ascending=False,
        )
    except Exception:
        importance_df = pd.DataFrame()

    return prediction_matrix, importance_df


def empirical_rank_transform(
    train_values,
    test_values,
):
    median = np.nanmedian(train_values)

    train_values = np.nan_to_num(
        train_values,
        nan=median,
    )
    test_values = np.nan_to_num(
        test_values,
        nan=median,
    )

    sorted_train = np.sort(train_values)

    train_rank = (
        np.searchsorted(
            sorted_train,
            train_values,
            side="right",
        )
        / len(train_values)
    )

    test_rank = (
        np.searchsorted(
            sorted_train,
            test_values,
            side="right",
        )
        / len(train_values)
    )

    return (
        train_rank.astype(np.float32),
        test_rank.astype(np.float32),
    )


def pair_prediction_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
):
    train_rank = {}
    val_rank = {}

    for col in PAIR_COLUMNS:
        train_rank[col], val_rank[col] = (
            empirical_rank_transform(
                train_features[col].to_numpy(dtype=float),
                val_features[col].to_numpy(dtype=float),
            )
        )

    target_values = target.to_numpy(dtype=float)
    pair_targets = []

    for first, second in combinations(
        PAIR_COLUMNS,
        2,
    ):
        distance = (
            np.abs(
                val_rank[first][:, None]
                - train_rank[first][None, :]
            )
            + np.abs(
                val_rank[second][:, None]
                - train_rank[second][None, :]
            )
        )

        nearest_idx = np.argmin(
            distance,
            axis=1,
        )

        pair_targets.append(
            target_values[nearest_idx]
        )

    return np.column_stack(pair_targets)


def nearest_neighbor_lookup(
    train_raw_frame: pd.DataFrame,
    val_raw_frame: pd.DataFrame,
    target: pd.Series,
):
    train_numeric = train_raw_frame[
        numeric_raw_columns
    ].copy()

    val_numeric = val_raw_frame[
        numeric_raw_columns
    ].copy()

    medians = train_numeric.median()

    train_numeric = train_numeric.fillna(
        medians
    )
    val_numeric = val_numeric.fillna(
        medians
    )

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_numeric
    )
    val_scaled = scaler.transform(
        val_numeric
    )

    tree = cKDTree(train_scaled)

    nearest_dist, nearest_idx = tree.query(
        val_scaled,
        k=1,
    )

    nearest_target = (
        target.to_numpy(dtype=float)[nearest_idx]
    )

    return nearest_dist, nearest_target


def apply_postprocess(
    prediction,
    round_decimals,
    override_threshold,
    nearest_dist,
    nearest_target,
):
    result = np.clip(
        prediction.copy(),
        0.0,
        1.0,
    )

    if round_decimals is not None:
        result = np.round(
            result,
            round_decimals,
        )

    if override_threshold is not None:
        mask = nearest_dist < override_threshold
        replacement = nearest_target[mask].copy()

        if round_decimals is not None:
            replacement = np.round(
                replacement,
                round_decimals,
            )

        result[mask] = np.clip(
            replacement,
            0.0,
            1.0,
        )

    return result


## 4. 혈압 피처 A/B 테스트


In [5]:
cv_splits = list(
    KFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED,
    ).split(X_raw)
)

bp_rows = []
bp_oof = {}

for config_name in BP_CONFIGS:
    oof_pred = np.zeros(
        len(y),
        dtype=float,
    )

    for fold, (train_idx, val_idx) in enumerate(
        cv_splits,
        start=1,
    ):
        train_fold_raw = (
            X_raw.iloc[train_idx]
            .reset_index(drop=True)
        )

        val_fold_raw = (
            X_raw.iloc[val_idx]
            .reset_index(drop=True)
        )

        y_train = (
            y.iloc[train_idx]
            .reset_index(drop=True)
        )

        bounds = compute_iqr_bounds(
            train_fold_raw,
            WINSORIZE_COLUMNS,
        )

        train_w = apply_winsorize(
            train_fold_raw,
            bounds,
        )

        val_w = apply_winsorize(
            val_fold_raw,
            bounds,
        )

        train_features = build_bp_feature_set(
            train_w,
            config_name,
        )

        val_features = build_bp_feature_set(
            val_w,
            config_name,
        )

        tree_matrix = tree_prediction_matrix(
            train_features,
            val_features,
            y_train,
            n_estimators=SCREEN_N_ESTIMATORS,
        )

        pred = np.quantile(
            tree_matrix,
            BASE_TREE_QUANTILE,
            axis=1,
        )

        pred = np.round(
            np.clip(pred, 0.0, 1.0),
            2,
        )

        oof_pred[val_idx] = pred

        print(
            f"[{config_name}] "
            f"Fold {fold}/{N_SPLITS} 완료"
        )

    mae = mean_absolute_error(
        y,
        oof_pred,
    )

    bp_rows.append({
        "config": config_name,
        "cv_mae": mae,
        "feature_count": build_bp_feature_set(
            X_raw.head(5),
            config_name,
        ).shape[1],
    })

    bp_oof[config_name] = oof_pred

    print(
        f"{config_name}: "
        f"CV MAE={mae:.8f}\n"
    )

bp_df = pd.DataFrame(bp_rows)

baseline_mae = float(
    bp_df.loc[
        bp_df["config"] == "baseline",
        "cv_mae",
    ].iloc[0]
)

bp_df["improvement_vs_baseline"] = (
    baseline_mae - bp_df["cv_mae"]
)

bp_df = (
    bp_df
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

display(bp_df)


[baseline] Fold 1/5 완료
[baseline] Fold 2/5 완료
[baseline] Fold 3/5 완료
[baseline] Fold 4/5 완료
[baseline] Fold 5/5 완료
baseline: CV MAE=0.14811333

[bp_ratio] Fold 1/5 완료
[bp_ratio] Fold 2/5 완료
[bp_ratio] Fold 3/5 완료
[bp_ratio] Fold 4/5 완료
[bp_ratio] Fold 5/5 완료
bp_ratio: CV MAE=0.14749667

[pulse_pressure] Fold 1/5 완료
[pulse_pressure] Fold 2/5 완료
[pulse_pressure] Fold 3/5 완료
[pulse_pressure] Fold 4/5 완료
[pulse_pressure] Fold 5/5 완료
pulse_pressure: CV MAE=0.14752667

[ratio_plus_pulse] Fold 1/5 완료
[ratio_plus_pulse] Fold 2/5 완료
[ratio_plus_pulse] Fold 3/5 완료
[ratio_plus_pulse] Fold 4/5 완료
[ratio_plus_pulse] Fold 5/5 완료
ratio_plus_pulse: CV MAE=0.14780000

[mean_arterial_pressure] Fold 1/5 완료
[mean_arterial_pressure] Fold 2/5 완료
[mean_arterial_pressure] Fold 3/5 완료
[mean_arterial_pressure] Fold 4/5 완료
[mean_arterial_pressure] Fold 5/5 완료
mean_arterial_pressure: CV MAE=0.14812333

[replace_bp_with_ratio_pulse] Fold 1/5 완료
[replace_bp_with_ratio_pulse] Fold 2/5 완료
[replace_bp_with_ratio_pulse

,config,cv_mae,feature_count,improvement_vs_baseline
0,bp_ratio,0.147497,19,0.000617
1,pulse_pressure,0.147527,19,0.000587
2,ratio_plus_pulse,0.147800,20,0.000313
3,baseline,0.148113,18,0.000000
4,mean_arterial_pressure,0.148123,19,-0.000010
5,replace_bp_with_ratio_pulse,0.148353,18,-0.000240


## 5. 최적 혈압 구성 선택 및 중요도 확인


In [6]:
best_bp_row = bp_df.iloc[0]

BEST_BP_CONFIG = str(
    best_bp_row["config"]
)

best_bp_mae = float(
    best_bp_row["cv_mae"]
)

MIN_REQUIRED_IMPROVEMENT = 0.00005

if (
    BEST_BP_CONFIG != "baseline"
    and baseline_mae - best_bp_mae
    < MIN_REQUIRED_IMPROVEMENT
):
    BEST_BP_CONFIG = "baseline"
    best_bp_mae = baseline_mae

print("선택된 혈압 피처 구성:", BEST_BP_CONFIG)
print("Baseline MAE:", baseline_mae)
print("선택 구성 MAE:", best_bp_mae)
print("개선 폭:", baseline_mae - best_bp_mae)

# 전체 데이터에서 importance는 참고용
full_bounds = compute_iqr_bounds(
    X_raw,
    WINSORIZE_COLUMNS,
)

full_w = apply_winsorize(
    X_raw,
    full_bounds,
)

full_features = build_bp_feature_set(
    full_w,
    BEST_BP_CONFIG,
)

_, importance_df = tree_prediction_matrix(
    full_features,
    full_features.iloc[:10].copy(),
    y,
    n_estimators=SCREEN_N_ESTIMATORS,
    return_importance=True,
)

if len(importance_df):
    display(importance_df.head(40))


선택된 혈압 피처 구성: bp_ratio
Baseline MAE: 0.14811333333333335
선택 구성 MAE: 0.14749666666666666
개선 폭: 0.000616666666666682


,feature,importance
24,numerical__cholesterol,0.025745
34,numerical__bmi__copy1,0.025333
41,numerical__height__copy2,0.025295
37,numerical__bmi__copy4,0.025275
36,numerical__bmi__copy3,0.025254
46,numerical__weight__copy2,0.025210
31,numerical__cholesterol_glucose_ratio,0.025207
30,numerical__bmi,0.025192
22,numerical__height,0.025189
45,numerical__weight__copy1,0.025159


## 6. 최적 혈압 구성에서 최종 탐색


In [7]:
fold_cache = []

for fold, (train_idx, val_idx) in enumerate(
    cv_splits,
    start=1,
):
    train_fold_raw = (
        X_raw.iloc[train_idx]
        .reset_index(drop=True)
    )

    val_fold_raw = (
        X_raw.iloc[val_idx]
        .reset_index(drop=True)
    )

    y_train = (
        y.iloc[train_idx]
        .reset_index(drop=True)
    )

    bounds = compute_iqr_bounds(
        train_fold_raw,
        WINSORIZE_COLUMNS,
    )

    train_w = apply_winsorize(
        train_fold_raw,
        bounds,
    )

    val_w = apply_winsorize(
        val_fold_raw,
        bounds,
    )

    train_features = build_bp_feature_set(
        train_w,
        BEST_BP_CONFIG,
    )

    val_features = build_bp_feature_set(
        val_w,
        BEST_BP_CONFIG,
    )

    tree_matrix = tree_prediction_matrix(
        train_features,
        val_features,
        y_train,
        n_estimators=FINAL_N_ESTIMATORS,
    )

    # Pair는 기존 baseline 구조 유지
    train_pair = add_baseline_features(
        train_w
    )

    val_pair = add_baseline_features(
        val_w
    )

    pair_matrix = pair_prediction_matrix(
        train_pair,
        val_pair,
        y_train,
    )

    nearest_dist, nearest_target = (
        nearest_neighbor_lookup(
            train_fold_raw,
            val_fold_raw,
            y_train,
        )
    )

    fold_cache.append({
        "val_idx": val_idx,
        "tree_matrix": tree_matrix,
        "pair_matrix": pair_matrix,
        "nearest_dist": nearest_dist,
        "nearest_target": nearest_target,
    })

    print(
        f"Fold {fold}/{N_SPLITS} 캐시 완료"
    )


Fold 1/5 캐시 완료
Fold 2/5 캐시 완료
Fold 3/5 캐시 완료
Fold 4/5 캐시 완료
Fold 5/5 캐시 완료


In [8]:
search_rows = []

for tree_q in TREE_QUANTILES:
    for pair_q in PAIR_QUANTILES:
        for pair_weight in PAIR_WEIGHTS:
            for decimals in ROUND_OPTIONS:
                for threshold in OVERRIDE_THRESHOLDS:
                    oof_pred = np.zeros(
                        len(y),
                        dtype=float,
                    )

                    for cache in fold_cache:
                        tree_pred = np.quantile(
                            cache["tree_matrix"],
                            tree_q,
                            axis=1,
                        )

                        pair_pred = np.quantile(
                            cache["pair_matrix"],
                            pair_q,
                            axis=1,
                        )

                        blended = (
                            (1.0 - pair_weight)
                            * tree_pred
                            + pair_weight
                            * pair_pred
                        )

                        final_pred = apply_postprocess(
                            blended,
                            decimals,
                            threshold,
                            cache["nearest_dist"],
                            cache["nearest_target"],
                        )

                        oof_pred[
                            cache["val_idx"]
                        ] = final_pred

                    mae = mean_absolute_error(
                        y,
                        oof_pred,
                    )

                    search_rows.append({
                        "cv_mae": mae,
                        "tree_q": tree_q,
                        "pair_q": pair_q,
                        "pair_weight": pair_weight,
                        "round_decimals": decimals,
                        "override_threshold": threshold,
                        "bp_config": BEST_BP_CONFIG,
                    })

search_df = (
    pd.DataFrame(search_rows)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

display(search_df.head(30))
print(
    "최적 최종 OOF MAE:",
    float(search_df.iloc[0]["cv_mae"]),
)


,cv_mae,tree_q,pair_q,pair_weight,round_decimals,override_threshold,bp_config
0,0.147483,0.54,0.48,0.25,2.0,0.20,bp_ratio
1,0.147483,0.54,0.48,0.25,2.0,0.18,bp_ratio
2,0.147483,0.54,0.48,0.25,2.0,0.15,bp_ratio
3,0.147506,0.54,0.46,0.25,NaN,0.15,bp_ratio
4,0.147506,0.54,0.46,0.25,NaN,0.18,bp_ratio
5,0.147506,0.54,0.46,0.25,NaN,0.20,bp_ratio
6,0.147512,0.54,0.48,0.25,NaN,0.20,bp_ratio
7,0.147512,0.54,0.48,0.25,NaN,0.18,bp_ratio
8,0.147512,0.54,0.48,0.25,NaN,0.15,bp_ratio
9,0.147517,0.52,0.48,0.15,2.0,0.15,bp_ratio


최적 최종 OOF MAE: 0.14748333333333333


## 7. 제출 후보 생성


In [9]:
def row_to_config(row):
    return {
        "tree_q": float(row["tree_q"]),
        "pair_q": float(row["pair_q"]),
        "pair_weight": float(row["pair_weight"]),
        "round_decimals": (
            None
            if pd.isna(row["round_decimals"])
            else int(row["round_decimals"])
        ),
        "override_threshold": (
            None
            if pd.isna(row["override_threshold"])
            else float(row["override_threshold"])
        ),
        "cv_mae": float(row["cv_mae"]),
    }


candidate_rows = [
    ("cv_best", search_df.iloc[0]),
]

safe_rows = search_df[
    (search_df["tree_q"] == 0.52)
    & (search_df["pair_q"] == 0.48)
    & (search_df["pair_weight"] == 0.20)
    & (search_df["round_decimals"] == 2)
]
if len(safe_rows):
    candidate_rows.append(
        ("safe", safe_rows.iloc[0])
    )

no_override_rows = search_df[
    search_df["override_threshold"].isna()
]
if len(no_override_rows):
    candidate_rows.append(
        (
            "no_override",
            no_override_rows.iloc[0],
        )
    )

tree_heavy_rows = search_df[
    search_df["pair_weight"]
    == min(PAIR_WEIGHTS)
]
if len(tree_heavy_rows):
    candidate_rows.append(
        (
            "tree_heavy",
            tree_heavy_rows.iloc[0],
        )
    )

pair_heavy_rows = search_df[
    search_df["pair_weight"]
    == max(PAIR_WEIGHTS)
]
if len(pair_heavy_rows):
    candidate_rows.append(
        (
            "pair_heavy",
            pair_heavy_rows.iloc[0],
        )
    )

FINAL_CONFIGS = []
seen = set()

for name, row in candidate_rows:
    config = row_to_config(row)

    key = (
        config["tree_q"],
        config["pair_q"],
        config["pair_weight"],
        config["round_decimals"],
        config["override_threshold"],
    )

    if key not in seen:
        seen.add(key)
        FINAL_CONFIGS.append(
            (name, config)
        )

FINAL_CONFIGS = FINAL_CONFIGS[:5]

print("최종 제출 후보")
for name, config in FINAL_CONFIGS:
    print(name, config)


최종 제출 후보
cv_best {'tree_q': 0.54, 'pair_q': 0.48, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.2, 'cv_mae': 0.14748333333333333}
safe {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.2, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14752}
no_override {'tree_q': 0.5, 'pair_q': 0.48, 'pair_weight': 0.15, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.14756000000000002}
tree_heavy {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.15, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14751666666666666}


In [10]:
if not sample_path.exists():
    submission_template = pd.DataFrame({
        ID_COLUMN: test_raw[ID_COLUMN],
        TARGET: 0.0,
    })
else:
    submission_template = pd.read_csv(
        sample_path
    )

bounds = compute_iqr_bounds(
    X_raw,
    WINSORIZE_COLUMNS,
)

train_w = apply_winsorize(
    X_raw,
    bounds,
)

test_w = apply_winsorize(
    X_test_raw,
    bounds,
)

train_features = build_bp_feature_set(
    train_w,
    BEST_BP_CONFIG,
)

test_features = build_bp_feature_set(
    test_w,
    BEST_BP_CONFIG,
)

tree_matrix_test = tree_prediction_matrix(
    train_features,
    test_features,
    y,
    n_estimators=FINAL_N_ESTIMATORS,
)

train_pair = add_baseline_features(
    train_w
)

test_pair = add_baseline_features(
    test_w
)

pair_matrix_test = pair_prediction_matrix(
    train_pair,
    test_pair,
    y,
)

nearest_dist_test, nearest_target_test = (
    nearest_neighbor_lookup(
        X_raw,
        X_test_raw,
        y,
    )
)

saved_paths = []

for rank, (name, config) in enumerate(
    FINAL_CONFIGS,
    start=1,
):
    tree_pred = np.quantile(
        tree_matrix_test,
        config["tree_q"],
        axis=1,
    )

    pair_pred = np.quantile(
        pair_matrix_test,
        config["pair_q"],
        axis=1,
    )

    blended = (
        (1.0 - config["pair_weight"])
        * tree_pred
        + config["pair_weight"]
        * pair_pred
    )

    final_pred = apply_postprocess(
        blended,
        config["round_decimals"],
        config["override_threshold"],
        nearest_dist_test,
        nearest_target_test,
    )

    submission = submission_template.copy()
    submission[TARGET] = final_pred

    filename = (
        f"submit_bp_v9_rank{rank}_{name}"
        f"_bp-{BEST_BP_CONFIG}.csv"
    )

    output_file = OUTPUT_DIR / filename

    submission.to_csv(
        output_file,
        index=False,
    )

    saved_paths.append(output_file)

    print("\nSaved:", output_file)
    print("Config:", config)
    display(submission.head())

print("\n생성 완료")
for path in saved_paths:
    print(path)



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_bp_v9_rank1_cv_best_bp-bp_ratio.csv
Config: {'tree_q': 0.54, 'pair_q': 0.48, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.2, 'cv_mae': 0.14748333333333333}


,ID,stress_score
0,TEST_0000,0.61
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.52
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_bp_v9_rank2_safe_bp-bp_ratio.csv
Config: {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.2, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14752}


,ID,stress_score
0,TEST_0000,0.61
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.49
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_bp_v9_rank3_no_override_bp-bp_ratio.csv
Config: {'tree_q': 0.5, 'pair_q': 0.48, 'pair_weight': 0.15, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.14756000000000002}


,ID,stress_score
0,TEST_0000,0.58
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.48
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_bp_v9_rank4_tree_heavy_bp-bp_ratio.csv
Config: {'tree_q': 0.52, 'pair_q': 0.48, 'pair_weight': 0.15, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14751666666666666}


,ID,stress_score
0,TEST_0000,0.61
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.48
4,TEST_0004,0.53



생성 완료
/Users/bitsaem/Desktop/stress_project/8_6/submit_bp_v9_rank1_cv_best_bp-bp_ratio.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_bp_v9_rank2_safe_bp-bp_ratio.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_bp_v9_rank3_no_override_bp-bp_ratio.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_bp_v9_rank4_tree_heavy_bp-bp_ratio.csv


## 확인할 결과

1. `bp_df`
   - 혈압비, 맥압, 둘의 조합 중 무엇이 가장 좋은지 확인
2. Feature importance
   - 새 혈압 피처가 실제로 어느 정도 사용되는지 확인
3. `search_df.head(30)`
   - Pair와 후처리까지 포함한 최종 OOF 최적값 확인
